# Proyecto RaSa: Primer entregable grupal

"""
Presentado por:

* DIANA CAROLINA LEAL MARTINEZ
* DIEGO ALEJANDRO FIGUEREDO SALAMANCA
* MARIA CAMILA REYES GIL
* SILVIA CARDENAS SANTOS

"""

```
Curso: Modelado de Datos y ETL

```


```
Universisda de los Andes Colombia

```
```
Año 2026
```



## Configuración de la conexión a la base de datos

In [2]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_CODESPACES = os.environ.get("CODESPACES") == "true"

if IN_COLAB or IN_CODESPACES:
    %pip install -q pyspark

if IN_COLAB:
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

ENVIRONMENT = (
    "colab" if IN_COLAB
    else "codespaces" if IN_CODESPACES
    else "local"
)

print(f"Entorno detectado: {ENVIRONMENT}")

from pyspark.sql import SparkSession
from pyspark.sql import functions
from pyspark.sql.types import StructType
from pyspark.sql.types import FloatType, StringType, IntegerType, DateType
from pyspark.sql.functions import udf, col, length, isnan, when, count
import pyspark.sql.functions as f
from datetime import datetime
from pyspark.sql import types as t

import matplotlib.pyplot as plt
import numpy as np

Entorno detectado: colab


In [3]:
## otras librerías
from pyspark.sql import functions as sf
from pyspark.sql.functions import col, count, when, trim, lit

### Configuración del controlador e inicio de sesión Spark

El conector JDBC de MySQL (Connector/J 9.7.0) se descarga la primera vez en `~/.miid4103/jars` (requiere internet) y lo reutilizan los demás notebooks.

In [4]:
from pathlib import Path
import urllib.request

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()
print(f'Conector JDBC: {MYSQL_JAR}')

Descargando MySQL Connector/J 9.7.0...
Conector JDBC: /root/.miid4103/jars/mysql-connector-j-9.7.0.jar


In [5]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
# Necesario en Windows si Python está en C:\Program Files
import pyspark
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))
# Spark 4.2 en Windows requiere HADOOP_HOME
if os.name == 'nt' and 'HADOOP_HOME' not in os.environ:
    hadoop_home = Path.home() / '.miid4103' / 'hadoop'
    (hadoop_home / 'bin').mkdir(parents=True, exist_ok=True)
    (hadoop_home / 'bin' / 'winutils.exe').touch()
    os.environ['HADOOP_HOME'] = str(hadoop_home)

spark = (SparkSession.builder
         .master('local[*]')
         .appName('MIID4103-NB02')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')    # comportamiento de Spark 3.5
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

## credenciales

In [6]:
db_host = '157.253.236.120'
db_port = 8080
db_user = 'DB_202615_'
db_psswd = ''
db_database = 'RaSaTransaccional'

if not db_user or not db_psswd:
    raise ValueError('Complete db_user y db_psswd antes de continuar')
jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'
db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_database}?{jdbc_opciones}'

# En Colab, suba los archivos a /content/
PATH = '/content/' if ENVIRONMENT == 'colab' else './'

OUT = Path('/content/salidas') if ENVIRONMENT == 'colab' else Path.home() / 'MIID4103' / 'salidas'
OUT.mkdir(parents=True, exist_ok=True)

## conexiona fuentes

In [7]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

## Bases de datos

### Areas de servicio

In [8]:
# Solución: Envolver la consulta entre paréntesis y darle un alias (AS t)
sql_fuente_areas_servicio = """
    (SELECT * FROM RaSaTransaccional.FuenteAreasDeServicio_Copia_E) AS t
"""

fuente_areas_servicio = obtener_dataframe_de_bd(db_connection_string, sql_fuente_areas_servicio, db_user, db_psswd)

# Verificar que cargó correctamente
fuente_areas_servicio.show(5)

+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|    464269.0|805.0|   577.0| 2017|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|    518117.0|185.0|  2801.0| 2018|
|          10132017|BlueOptions16842F...|        12031|     Duval County|   Florida|    999935.0|774.0|  1292.0| 2017|
|         101982018|New Jersey - Medi...|        34003|    Bergen County|New Jersey|    953819.0|234.0|  4076.0| 2018|
|         102012017|New Jersey - Medi...|        34021|    Mercer County|New Jersey|    385898.0|226.0|  1708.0| 2017|
+------------------+--------------------+-------

### Tipos de Beneficios

In [9]:
## tipos de beneficio
sql_fuente_tipos_beneficio ="""(SELECT * FROM RaSaTransaccional.FuenteTiposBeneficio_Copia_E) as b """
fuente_tipos_beneficio = obtener_dataframe_de_bd(db_connection_string, sql_fuente_tipos_beneficio, db_user, db_psswd)

In [10]:
# Verificar que cargó correctamente
fuente_tipos_beneficio.show(5)

+-----------------+--------------------+--------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|IdTipoBeneficio_T|              Nombre|     UnidadDelLimite|EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|Fecha|
+-----------------+--------------------+--------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|              565|Nutritional Couns...|                    |   No|                  Yes|                     No|                                      No|                                    Yes| 2017|
|              795|Rehabilitative Sp...|Days per Benefit ...|  Yes|                  Yes|                    Yes|                                      No|                                    Yes| 2

### Planes de Beneficios

In [11]:
sql_fuente_planes_beneficio = """(SELECT * FROM RaSaTransaccional.FuentePlanesBeneficio_Copia_E) as p """
fuente_planes_beneficio = obtener_dataframe_de_bd(db_connection_string, sql_fuente_planes_beneficio, db_user, db_psswd)
fuente_planes_beneficio.show(5)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|              640|          10382017|                       34|                         27|                3|16842FL0070128-03|2017-12-31|        16842|          0|           50|            35|
|              225|          31512017|                      238|                         45|                2|29418TX0140002-04|2017-12-31|        29418|          0|            0|          NULL|
|              190|      

### Condiciones De pago

In [12]:
sql_condiciones_pago = """(SELECT * FROM RaSaTransaccional.FuenteCondicionesDePago_Copia_E) as c """
condiciones_pago = obtener_dataframe_de_bd(db_connection_string, sql_condiciones_pago, db_user, db_psswd)
condiciones_pago.show(5)

+---------------------+--------------------+--------+
|IdCondicionesDePago_T|         Descripcion|    Tipo|
+---------------------+--------------------+--------+
|                  187|Copay with deduct...|  Copago|
|                  204|       Copay per Day|  Copago|
|                   45|         Coinsurance|Coseguro|
|                   85|Copay per Day bef...|  Copago|
|                   18|No Charge after d...|Coseguro|
+---------------------+--------------------+--------+
only showing top 5 rows


#

## Entregable 2

Análisis de calidad de datos: Revise las 4 dimensiones de calidad vistas en el curso en las fuentes compartidas y saque conclusiones

#### Completitud

En el perfilamiento de los datos se había hecho un análisis de los datos nulos, solo para recordar se obtuvo que:

| Fuente| Cantidad de registro | Variables con Nulos|
| --- | --- | --- |
| Areas de servicio | 188815 | IdAreaDeServicio_T, IdGeografia_T, Area, Densidad, Fecha |  
| Tipo de beneficios | 849  |  |
|  Planes Beneficio| 36036 |  IdTipoBeneficio_T,IdAreaDeServicio_T, cantidadLimite|
|  Condiciones de pago | 31 |  |

##### Fuente de areas de servicio

A continucación se revisa si hay datos nulos, datos NAN o datos con espacios (trim " ")

In [13]:
fuente_areas_servicio.select([count(when(col(c).isNull() | (isnan(c) if dict(fuente_areas_servicio.dtypes)[c] in ["double", "float"] else col(c).isNull()) | (trim(col(c)) == ""), 1)).alias(c) for c in fuente_areas_servicio.columns]).show()

+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|Condado|Estado|PoblacionAct|Area|Densidad|Fecha|
+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+
|              6288|                   0|         6378|      0|     0|           0|2407|    2407|    0|
+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+



In [14]:
# Contar nulos por columna y mostrar el resultado
fuente_areas_servicio.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in fuente_areas_servicio.columns
]).show()

+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|Condado|Estado|PoblacionAct|Area|Densidad|Fecha|
+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+
|              6288|                   0|         6378|      0|     0|           0|2407|    2407|    0|
+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+



>Se registran 6288 datos en el área de servicios, 6378 identificadores geográficos, 2407 registros de área y 2407 valores faltantes en el área de densidad.

In [15]:
fuente_areas_servicio.filter(col("IdGeografia_T").isNull()).show()

+------------------+--------------------+-------------+------------------+----------+------------+------+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|           Condado|    Estado|PoblacionAct|  Area|Densidad|Fecha|
+------------------+--------------------+-------------+------------------+----------+------------+------+--------+-----+
|          28382018|Indiana28856IN018...|         NULL|      Allen County|   Indiana|    388608.0| 657.0|   591.0| 2018|
|          52712017|Oakland40047MI007...|         NULL|     Macomb County|  Michigan|    876792.0| 570.0|  1538.0| 2017|
|            442017|Wyoming11269WY008...|         NULL|   Big Horn County|   Wyoming|     11632.0|3137.0|     4.0| 2017|
|          46352018|Elite37833WI05100...|         NULL|  Marquette County| Wisconsin|     15792.0| 455.0|    35.0| 2018|
|          31542017|Ambetter from Sup...|         NULL|    Bandera County|     Texas|     21565.0| 792.0|    27.0| 2017|
|          43692018|Blue Choice 

In [16]:
fuente_areas_servicio.filter(col("IdAreaDeServicio_T").isNotNull()).show()

+------------------+--------------------+-------------+-----------------+----------+------------+------+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct|  Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+------+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|    464269.0| 805.0|   577.0| 2017|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|    518117.0| 185.0|  2801.0| 2018|
|          10132017|BlueOptions16842F...|        12031|     Duval County|   Florida|    999935.0| 774.0|  1292.0| 2017|
|         101982018|New Jersey - Medi...|        34003|    Bergen County|New Jersey|    953819.0| 234.0|  4076.0| 2018|
|         102012017|New Jersey - Medi...|        34021|    Mercer County|New Jersey|    385898.0| 226.0|  1708.0| 2017|
|         102012017|New Jersey - Medi...

> En el análisis de los identificadores, se observó que algunos nombres de áreas de servicio se repiten bajo un mismo ID (por ejemplo, BlueOptions16842F... corresponde al ID 10242017); esto permite inferir que existen áreas de servicio que aún no cuentan con una identificación única.

In [17]:
fuente_areas_servicio.filter(col("IdGeografia_T").isNull()).show()

+------------------+--------------------+-------------+------------------+----------+------------+------+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|           Condado|    Estado|PoblacionAct|  Area|Densidad|Fecha|
+------------------+--------------------+-------------+------------------+----------+------------+------+--------+-----+
|          28382018|Indiana28856IN018...|         NULL|      Allen County|   Indiana|    388608.0| 657.0|   591.0| 2018|
|          52712017|Oakland40047MI007...|         NULL|     Macomb County|  Michigan|    876792.0| 570.0|  1538.0| 2017|
|            442017|Wyoming11269WY008...|         NULL|   Big Horn County|   Wyoming|     11632.0|3137.0|     4.0| 2017|
|          46352018|Elite37833WI05100...|         NULL|  Marquette County| Wisconsin|     15792.0| 455.0|    35.0| 2018|
|          31542017|Ambetter from Sup...|         NULL|    Bandera County|     Texas|     21565.0| 792.0|    27.0| 2017|
|          43692018|Blue Choice 

In [18]:
fuente_areas_servicio.filter(col("IdGeografia_T").isNotNull()).show()

+------------------+--------------------+-------------+-----------------+----------+------------+------+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct|  Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+------+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|    464269.0| 805.0|   577.0| 2017|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|    518117.0| 185.0|  2801.0| 2018|
|          10132017|BlueOptions16842F...|        12031|     Duval County|   Florida|    999935.0| 774.0|  1292.0| 2017|
|         101982018|New Jersey - Medi...|        34003|    Bergen County|New Jersey|    953819.0| 234.0|  4076.0| 2018|
|         102012017|New Jersey - Medi...|        34021|    Mercer County|New Jersey|    385898.0| 226.0|  1708.0| 2017|
|         102012017|New Jersey - Medi...

> A partir de las tablas anteriores, se deduce que existen condados que aún no han sido identificados.

In [19]:
fuente_areas_servicio.filter(col("Area").isNull()).show()

+------------------+--------------------+-------------+--------------------+--------+------------+----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|             Condado|  Estado|PoblacionAct|Area|Densidad|Fecha|
+------------------+--------------------+-------------+--------------------+--------+------------+----+--------+-----+
|         108722018|Service Area 0019...|        51510|     Alexandria City|Virginia|    158309.0|NULL|    NULL| 2018|
|          18492018|OHP Service Area2...|        51650|        Hampton City|Virginia|    135169.0|NULL|    NULL| 2018|
|          58882018|Virginia-Richmond...|        51760|       Richmond City|Virginia|    229233.0|NULL|    NULL| 2018|
|          61472018|Virginia-Richmond...|        51730|     Petersburg City|Virginia|     30791.0|NULL|    NULL| 2018|
|          57082018|Virginia-Richmond...|        51670|       Hopewell City|Virginia|     22500.0|NULL|    NULL| 2018|
|          58432018|Virginia-Richmond...|       

In [20]:
fuente_areas_servicio.filter(
    (col("Area").isNull()) & (col("Densidad").isNotNull())
).show()

+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|Condado|Estado|PoblacionAct|Area|Densidad|Fecha|
+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+
+------------------+--------------------+-------------+-------+------+------------+----+--------+-----+



> A partir de la cantidad de valores ausentes y la información expuesta en la gráfica anterior, se determina que los datos faltantes de "Área" y "Densidad" corresponden a los mismos registros (filas).

##### Planes beneficio

In [21]:
fuente_planes_beneficio.select([count(when(col(c).isNull() | (isnan(c) if dict(fuente_planes_beneficio.dtypes)[c] in ["double", "float"] else col(c).isNull()) | (trim(col(c)) == ""), 1)).alias(c) for c in fuente_planes_beneficio.columns]).show()

+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|IdPlan_T|Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+
|             2086|              2041|                        0|                          0|                0|    2053|    0|            0|          0|            0|         30571|
+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+



In [22]:
# contar nulos por columna y mostrar el resultado
fuente_planes_beneficio.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in fuente_planes_beneficio.columns
]).show()

+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|IdPlan_T|Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+
|             2086|              2041|                        0|                          0|                0|       0|    0|            0|          0|            0|         30571|
+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+



In [23]:
# contar si està vacía por columna y mostrar el resultado
fuente_planes_beneficio.select([ count(when(trim(col(c)) == "", c)).alias(c)
    if dict(fuente_planes_beneficio.dtypes).get(c) == "string"
    else lit(0).alias(c)
    for c in fuente_planes_beneficio.columns]).show()

+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|IdPlan_T|Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+
|                0|                 0|                        0|                          0|                0|    2053|    0|            0|          0|            0|             0|
+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+-----+-------------+-----------+-------------+--------------+



> En la fuente Planes de beneficios se identificaron 2086 valores faltantes en la variable IdTiposBeneficio, 2041 en IdAreaDeServicio_T y 30571 en Cantidad límite. Adicionalmente, en un análisis posterior se determinó la existencia de 2053 registros vacíos en Idplan_T.

In [24]:
fuente_planes_beneficio.filter(col("IdTipoBeneficio_T").isNull()).show()

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|             NULL|         107352017|                      238|                         45|                2|93332GA0710044-00|2017-12-31|        93332|          0|            0|          NULL|
|             NULL|          85992018|                      238|                         45|                2|71933PA0020003-00|2018-12-31|        71933|          0|            0|             1|
|             NULL|      

In [25]:
fuente_planes_beneficio.filter(col("IdAreaDeServicio_T").isNull()).show()

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|              590|              NULL|                      238|                         45|                2|33602TX0770036-05|2017-12-31|        33602|          0|            0|          NULL|
|              490|              NULL|                       34|                         45|                3|54192IN0010010-00|2018-12-31|        54192|          0|          100|          NULL|
|              790|      

In [26]:
fuente_planes_beneficio.filter(trim(col("IdPlan_T")) == "").show()

+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+--------+----------+-------------+-----------+-------------+--------------+
|             1000|          38532017|                       34|                         36|                2|        |2017-12-31|        33602|          0|            0|          NULL|
|              865|          52222017|                      238|                         45|                2|        |2017-12-31|        40047|          0|            0|            45|
|             1055|          29072017|                       34|      

> Los datos nulos no parecen guardar una relación con otras variables de la misma tabla, al igual que los datos vacíos.

#### Validez

Previamente, durante la etapa de perfilamiento de datos, se identificaron los tipos de variables y se realizó un análisis descriptivo. A continuación, de manera complementaria, se validarán los formatos de algunas variables y se detectarán aquellos registros que incumplan los criterios de validez

##### Fuente de datos

In [27]:
import re
regex_fecha = r"^\d{4}-\d{2}-\d{2}$"

In [28]:
fuente_areas_servicio.select('Fecha').filter(~col('Fecha').rlike(regex_fecha)).show()

+-----+
|Fecha|
+-----+
| 2017|
| 2018|
| 2017|
| 2018|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2018|
| 2018|
| 2017|
| 2018|
| 2017|
| 2017|
| 2018|
| 2018|
| 2017|
+-----+
only showing top 20 rows


In [29]:
print(f"Registros con fecha inválida: {fuente_areas_servicio.filter(~col('Fecha').rlike(regex_fecha)).count()}")
print(f"total de registros: {fuente_areas_servicio.select('Fecha').count()}")

Registros con fecha inválida: 188815
total de registros: 188815


> Todos los registos de fecha contienen contienen el formato de año

In [30]:
# contar el número de registos por fecha
fuente_areas_servicio.groupBy('Fecha').count().orderBy('Fecha').show()

+-----+-----+
|Fecha|count|
+-----+-----+
| 1800| 6282|
| 2017|99211|
| 2018|83322|
+-----+-----+



> Resulta llamativo que existan 6282 registros con fecha de creación en el año 1800, un valor atípico muy distante del resto de las fechas registradas.

##### Condicones de pago

In [31]:
# agrupar condiciones de pago por IdCondicionDePago_T y contar el número de registros por cada condición
condiciones_pago.groupBy('Tipo').count().orderBy('count', ascending=False).show()

+-------------+-----+
|         Tipo|count|
+-------------+-----+
|       Copago|   19|
|     Coseguro|    7|
|          NaN|    2|
|SinTipoCopago|    1|
|  Coseguridad|    1|
|     Copagado|    1|
+-------------+-----+



> Según el diccionario de datos, los valores permitidos para la variable "Tipo" son Copago, Coaseguro y Anticipado. Sin embargo, no se encontró ningún registro con el valor Anticipado; en su lugar, aparecen categorías no especificadas como Cooseguridad y Copagado.

##### Planes de Beneficio

In [32]:
print(f"Registros con fecha inválida: {fuente_planes_beneficio.filter(~col('Fecha').rlike(regex_fecha)).count()}")
print(f"total de registros: {fuente_planes_beneficio.select('Fecha').count()}")


Registros con fecha inválida: 2056
total de registros: 36036


In [33]:
fuente_planes_beneficio.select('Fecha').filter(~col('Fecha').rlike(regex_fecha)).show()

+-----------+
|      Fecha|
+-----------+
|Dec 31,2017|
|Dec 31,2017|
|Dec 31,2018|
|Dec 31,2017|
|Dec 31,2017|
|Dec 31,2018|
|Dec 31,2017|
|Dec 31,2018|
|Dec 31,2017|
|Dec 31,2018|
|Dec 31,2017|
|Dec 31,2018|
|Dec 31,2018|
|Dec 31,2018|
|Dec 31,2017|
|Dec 31,2018|
|Dec 31,2018|
|Dec 31,2018|
|Dec 31,2018|
|Dec 31,2018|
+-----------+
only showing top 20 rows


In [34]:
fuente_planes_beneficio.select('Fecha').filter(col('Fecha').rlike(regex_fecha)).show()

+----------+
|     Fecha|
+----------+
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2018-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
|2018-12-31|
|2018-12-31|
|2017-12-31|
|2018-12-31|
|2017-12-31|
|2017-12-31|
|2017-12-31|
+----------+
only showing top 20 rows


> La variable de fecha de la fuente de planes de beneficio no tiene un formato único.

##### Tipo de beneficio

In [35]:
print(f"Registros con fecha inválida: {fuente_tipos_beneficio.filter(~col('Fecha').rlike(regex_fecha)).count()}")
print(f"total de registros: {fuente_tipos_beneficio.select('Fecha').count()}")


Registros con fecha inválida: 849
total de registros: 849


In [36]:
fuente_tipos_beneficio.select('Fecha').filter(~col('Fecha').rlike(regex_fecha)).show()

+-----+
|Fecha|
+-----+
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2017|
| 2018|
| 2018|
| 2017|
| 2017|
| 2018|
| 2017|
| 2017|
| 2018|
+-----+
only showing top 20 rows


> Todos los formatos de tipo de beneficio corresponden a un año.

In [37]:
fuente_tipos_beneficio.groupBy('Fecha').count().orderBy('Fecha').show()

+-----+-----+
|Fecha|count|
+-----+-----+
| 2017|  672|
| 2018|  177|
+-----+-----+



> Los datos corresponden a los años 2017 y 2018, con una clara concentración de registros en 2017.

#### Unicidad

En el **Entregable 1 (Perfilamiento)** se identificó la presencia de **filas exactamente duplicadas** en todas las fuentes:
* **F1 (`FuenteAreasDeServicio`):** 43,573 filas excedentes (de 188,815 totales $\rightarrow$ 145,242 filas distintas).
* **F2 (`FuenteTiposBeneficio`):** 271 filas excedentes (de 849 totales $\rightarrow$ 578 filas distintas).
* **F3 (`FuentePlanesBeneficio`):** 8,493 filas excedentes (de 36,036 totales $\rightarrow$ 27,543 filas distintas).
* **F4 (`FuenteCondicionesDePago`):** 7 filas excedentes (de 31 totales $\rightarrow$ 24 filas distintas).

A continuación, se evalúa la **unicidad de entidad y de llave primaria/compuesta** *después* de descartar los duplicados exactos, con el fin de detectar **colisiones de llave** (un mismo identificador asociado a atributos diferentes).

In [38]:
# 1. Eliminación de duplicados exactos de fila completa para evaluar unicidad de llaves
f1_sin_dup = fuente_areas_servicio.dropDuplicates()
f2_sin_dup = fuente_tipos_beneficio.dropDuplicates()
f3_sin_dup = fuente_planes_beneficio.dropDuplicates()
f4_sin_dup = condiciones_pago.dropDuplicates()

print("=== 1. UNICIDAD EN F4 (CONDICIONES DE PAGO) TRAS QUITAR DUPLICADOS EXACTOS ===")
print(f"Filas distintas en F4: {f4_sin_dup.count()} | IdCondicionesDePago_T únicos: {f4_sin_dup.select('IdCondicionesDePago_T').distinct().count()}")
ids_rep_f4 = f4_sin_dup.groupBy("IdCondicionesDePago_T").count().filter(col("count") > 1).select("IdCondicionesDePago_T")
print("Registros en F4 donde el mismo IdCondicionesDePago_T tiene diferentes atributos:")
f4_sin_dup.join(ids_rep_f4, on="IdCondicionesDePago_T", how="inner").orderBy("IdCondicionesDePago_T").show(truncate=False)

print("=== 2. UNICIDAD EN F2 (TIPOS DE BENEFICIO) POR (IdTipoBeneficio_T, Fecha) ===")
total_f2_limpio = f2_sin_dup.count()
llave_f2_unica = f2_sin_dup.select("IdTipoBeneficio_T", "Fecha").distinct().count()
print(f"Filas distintas en F2: {total_f2_limpio} | Combinaciones únicas (IdTipoBeneficio_T, Fecha): {llave_f2_unica}")
print(f"Colisiones de llave (mismo beneficio y año con atributos diferentes): {total_f2_limpio - llave_f2_unica}")
ids_rep_f2 = f2_sin_dup.groupBy("IdTipoBeneficio_T", "Fecha").count().filter(col("count") > 1).select("IdTipoBeneficio_T", "Fecha")
f2_sin_dup.join(ids_rep_f2, on=["IdTipoBeneficio_T", "Fecha"], how="inner").orderBy("IdTipoBeneficio_T", "Fecha").show(10, truncate=False)

print("=== 3. UNICIDAD EN F1 (ÁREAS DE SERVICIO) POR (IdAreaDeServicio_T, IdGeografia_T, Fecha) ===")
f1_no_nulos = f1_sin_dup.filter(col("IdAreaDeServicio_T").isNotNull() & col("IdGeografia_T").isNotNull())
total_f1_val = f1_no_nulos.count()
llave_f1_unica = f1_no_nulos.select("IdAreaDeServicio_T", "IdGeografia_T", "Fecha").distinct().count()
print(f"Filas distintas sin IDs nulos en F1: {total_f1_val} | Llaves únicas (IdArea, IdGeografia, Fecha): {llave_f1_unica}")
print(f"Colisiones de llave en F1 (mismo área, condado y año con población/área diferente): {total_f1_val - llave_f1_unica}")

print("=== 4. UNICIDAD EN F3 (PLANES DE BENEFICIO) POR LLAVE COMPUESTA DEL NEGOCIO ===")
llave_f3 = ["IdPlan_T", "IdTipoBeneficio_T", "IdAreaDeServicio_T", "IdNivelServicio_T", "IdCondicionDePagoCopago_T", "IdCondicionDePagoCoseguro_T", "Fecha"]
f3_no_nulos = f3_sin_dup.filter(col("IdTipoBeneficio_T").isNotNull() & col("IdAreaDeServicio_T").isNotNull() & (trim(col("IdPlan_T")) != ""))
total_f3_val = f3_no_nulos.count()
llave_f3_unica = f3_no_nulos.select(llave_f3).distinct().count()
print(f"Filas distintas válidas en F3: {total_f3_val} | Combinaciones únicas de llave compuesta: {llave_f3_unica}")
print(f"Colisiones de llave en F3 (misma llave pero distinto copago/coseguro/límite): {total_f3_val - llave_f3_unica}")

=== 1. UNICIDAD EN F4 (CONDICIONES DE PAGO) TRAS QUITAR DUPLICADOS EXACTOS ===
Filas distintas en F4: 24 | IdCondicionesDePago_T únicos: 23
Registros en F4 donde el mismo IdCondicionesDePago_T tiene diferentes atributos:
+---------------------+-------------------------------+------+
|IdCondicionesDePago_T|Descripcion                    |Tipo  |
+---------------------+-------------------------------+------+
|204                  |Copay per Stay after deductible|Copago|
|204                  |Copay per Day                  |Copago|
+---------------------+-------------------------------+------+

=== 2. UNICIDAD EN F2 (TIPOS DE BENEFICIO) POR (IdTipoBeneficio_T, Fecha) ===
Filas distintas en F2: 578 | Combinaciones únicas (IdTipoBeneficio_T, Fecha): 218
Colisiones de llave (mismo beneficio y año con atributos diferentes): 360
+-----------------+-----+-----------------------------------------------+------------------------+-----+---------------------+-----------------------+----------------

#### Consistencia

En esta dimensión se evalúa la coherencia lógica e integridad de los datos en tres niveles:
1. **Consistencia intra-tabla:** Coherencia entre indicadores categóricos (`TieneLimiteCuantitativo` vs `UnidadDelLimite`), relación matemática (`PoblacionAct / Area` vs `Densidad`), valores negativos o extremos y coherencia entre el año codificado en los IDs y el campo `Fecha`.
2. **Consistencia con las reglas de negocio de RaSA:** Verificación de límites cuantitativos mayores a cero en planes (`F2` vs `F3`), topes máximos de copago ($3,300) y coseguro (100%) en 2018, y conteo de condiciones de pago tras homologar etiquetas equivalentes (`Copagado`, `Coseguridad`).
3. **Integridad referencial inter-tabla:** Cuantificación de registros huérfanos en `F3` cuyos identificadores (`IdAreaDeServicio_T`, `IdTipoBeneficio_T`, `IdCondicionDePago`) no existen en las tablas maestras `F1`, `F2` y `F4`.

In [39]:
print("=== 1. CONSISTENCIA INTRA-TABLA EN F1: VALORES NEGATIVOS, EXTREMOS Y FÓRMULA DE DENSIDAD ===")
print(f"Registros con Area <= 0 (incluye negativos como -24707): {fuente_areas_servicio.filter(col('Area') <= 0).count()}")
print(f"Registros con PoblacionAct > 20,000,000 (supera al condado más poblado de EE.UU.): {fuente_areas_servicio.filter(col('PoblacionAct') > 20000000).count()}")

f1_densidad = fuente_areas_servicio.filter((col("Area") > 0) & col("Densidad").isNotNull()) \
    .withColumn("Densidad_Esperada", sf.round(col("PoblacionAct") / col("Area"), 0)) \
    .withColumn("Diff_Densidad", sf.abs(col("Densidad") - col("Densidad_Esperada")))
print(f"Registros donde Densidad no concuerda con (PoblacionAct / Area) [diferencia > 5]: {f1_densidad.filter(col('Diff_Densidad') > 5).count()}")

f1_anio_id = fuente_areas_servicio.filter(col("IdAreaDeServicio_T").isNotNull()) \
    .withColumn("Anio_Sufijo_ID", sf.substring(col("IdAreaDeServicio_T").cast("string"), -4, 4))
print(f"Registros en F1 donde el año del sufijo de IdAreaDeServicio_T difiere de Fecha: {f1_anio_id.filter(col('Anio_Sufijo_ID') != col('Fecha').cast('string')).count()}")

print("\n=== 2. CONSISTENCIA EN F2: VARIABLES BOOLEANAS NO ESTANDARIZADAS Y UNIDAD DEL LÍMITE ===")
f2_norm = fuente_tipos_beneficio.withColumn(
    "Limite_Norm",
    when(col("TieneLimiteCuantitativo").isin("Yes", "Si"), "Si")
    .when(col("TieneLimiteCuantitativo").isin("No", "Nein"), "No")
    .otherwise("Otro")
)
incons_f2_unidad = f2_norm.filter(
    ((col("Limite_Norm") == "Si") & (col("UnidadDelLimite").isNull() | (trim(col("UnidadDelLimite")) == ""))) |
    ((col("Limite_Norm") == "No") & (col("UnidadDelLimite").isNotNull() & (trim(col("UnidadDelLimite")) != "")))
).count()
print(f"Registros en F2 con contradicción entre TieneLimiteCuantitativo y UnidadDelLimite: {incons_f2_unidad}")

print("\n=== 3. REGLA DE NEGOCIO #2: LÍMITE CUANTITATIVO ENTRE F2 Y F3 ===")
ids_con_limite = f2_norm.filter(col("Limite_Norm") == "Si").select("IdTipoBeneficio_T").distinct()
ids_sin_limite = f2_norm.filter(col("Limite_Norm") == "No").select("IdTipoBeneficio_T").distinct()

f3_con_lim = fuente_planes_beneficio.join(ids_con_limite, on="IdTipoBeneficio_T", how="inner")
falla_lim_si = f3_con_lim.filter(col("cantidadLimite").isNull() | (col("cantidadLimite") <= 0)).count()
print(f"Planes en F3 cuyo beneficio exige límite cuantitativo ('Yes'/'Si'), pero cantidadLimite es NULL o <= 0: {falla_lim_si} de {f3_con_lim.count()}")

f3_sin_lim = fuente_planes_beneficio.join(ids_sin_limite, on="IdTipoBeneficio_T", how="inner")
falla_lim_no = f3_sin_lim.filter(col("cantidadLimite").isNotNull() & (col("cantidadLimite") > 0)).count()
print(f"Planes en F3 cuyo beneficio NO tiene límite ('No'/'Nein'), pero reportan cantidadLimite > 0: {falla_lim_no} de {f3_sin_lim.count()}")

print("\n=== 4. REGLA DE NEGOCIO #5 Y FECHAS EN F3: TOPES DE COPAGO/COSEGURO Y AÑO 1920 ===")
f3_anios = fuente_planes_beneficio.withColumn("Anio", sf.regexp_extract(col("Fecha"), r"(\d{4})", 1))
f3_anios.groupBy("Anio").agg(
    sf.count("*").alias("Registros"),
    sf.countDistinct(when(trim(col("IdPlan_T")) != "", col("IdPlan_T"))).alias("Planes_Unicos"),
    sf.max("valorCopago").alias("Max_Copago"),
    sf.count(when(col("valorCopago") > 3300, 1)).alias("Copago_Mayor_3300"),
    sf.max("valorCoseguro").alias("Max_Coseguro"),
    sf.count(when(col("valorCoseguro") > 100, 1)).alias("Coseguro_Mayor_100")
).orderBy("Anio").show()

=== 1. CONSISTENCIA INTRA-TABLA EN F1: VALORES NEGATIVOS, EXTREMOS Y FÓRMULA DE DENSIDAD ===
Registros con Area <= 0 (incluye negativos como -24707): 6177
Registros con PoblacionAct > 20,000,000 (supera al condado más poblado de EE.UU.): 6142
Registros donde Densidad no concuerda con (PoblacionAct / Area) [diferencia > 5]: 6182
Registros en F1 donde el año del sufijo de IdAreaDeServicio_T difiere de Fecha: 6282

=== 2. CONSISTENCIA EN F2: VARIABLES BOOLEANAS NO ESTANDARIZADAS Y UNIDAD DEL LÍMITE ===
Registros en F2 con contradicción entre TieneLimiteCuantitativo y UnidadDelLimite: 33

=== 3. REGLA DE NEGOCIO #2: LÍMITE CUANTITATIVO ENTRE F2 Y F3 ===
Planes en F3 cuyo beneficio exige límite cuantitativo ('Yes'/'Si'), pero cantidadLimite es NULL o <= 0: 14471 de 19322
Planes en F3 cuyo beneficio NO tiene límite ('No'/'Nein'), pero reportan cantidadLimite > 0: 4774 de 31844

=== 4. REGLA DE NEGOCIO #5 Y FECHAS EN F3: TOPES DE COPAGO/COSEGURO Y AÑO 1920 ===
+----+---------+-------------+--

In [40]:
print("=== 5. INTEGRIDAD REFERENCIAL (CRUCE DE F3 CON F1, F2 Y F4) ===")

areas_f1 = fuente_areas_servicio.filter(col("IdAreaDeServicio_T").isNotNull()).select("IdAreaDeServicio_T").distinct()
f3_areas_no_nulas = fuente_planes_beneficio.filter(col("IdAreaDeServicio_T").isNotNull())
huerfanos_f1_filas = f3_areas_no_nulas.join(areas_f1, on="IdAreaDeServicio_T", how="left_anti").count()
huerfanos_f1_ids = f3_areas_no_nulas.select("IdAreaDeServicio_T").distinct().join(areas_f1, on="IdAreaDeServicio_T", how="left_anti").count()
print(f"F3 -> F1 (Áreas): {huerfanos_f1_filas} filas en F3 ({huerfanos_f1_ids} IDs de área distintos) NO existen en F1.")

beneficios_f2 = fuente_tipos_beneficio.filter(col("IdTipoBeneficio_T").isNotNull()).select("IdTipoBeneficio_T").distinct()
f3_ben_no_nulos = fuente_planes_beneficio.filter(col("IdTipoBeneficio_T").isNotNull())
huerfanos_f2_filas = f3_ben_no_nulos.join(beneficios_f2, on="IdTipoBeneficio_T", how="left_anti").count()
huerfanos_f2_ids = f3_ben_no_nulos.select("IdTipoBeneficio_T").distinct().join(beneficios_f2, on="IdTipoBeneficio_T", how="left_anti").count()
print(f"F3 -> F2 (Beneficios): {huerfanos_f2_filas} filas en F3 ({huerfanos_f2_ids} IDs de beneficio distintos) NO existen en F2.")

f4_norm = condiciones_pago.withColumn(
    "Tipo_Homologado",
    when(col("Tipo").isin("Copago", "Copagado"), "Copago")
    .when(col("Tipo").isin("Coseguro", "Coseguridad"), "Coseguro")
    .otherwise(col("Tipo"))
)
print("\nConteo de IDs únicos en F4 tras homologar 'Copagado'->'Copago' y 'Coseguridad'->'Coseguro':")
f4_norm.groupBy("Tipo_Homologado").agg(sf.countDistinct("IdCondicionesDePago_T").alias("IDs_Unicos")).show()

ids_f4 = condiciones_pago.select("IdCondicionesDePago_T").distinct()
huerfanos_copago = fuente_planes_beneficio.join(ids_f4, fuente_planes_beneficio["IdCondicionDePagoCopago_T"] == ids_f4["IdCondicionesDePago_T"], how="left_anti").count()
huerfanos_coseguro = fuente_planes_beneficio.join(ids_f4, fuente_planes_beneficio["IdCondicionDePagoCoseguro_T"] == ids_f4["IdCondicionesDePago_T"], how="left_anti").count()
print(f"F3 -> F4: Filas en F3 con IdCondicionDePagoCopago_T inexistente en F4: {huerfanos_copago}")
print(f"F3 -> F4: Filas en F3 con IdCondicionDePagoCoseguro_T inexistente en F4: {huerfanos_coseguro}")

incons_prov = fuente_planes_beneficio.filter(trim(col("IdPlan_T")) != "") \
    .withColumn("Prefijo_Plan", sf.substring(col("IdPlan_T"), 1, 5)) \
    .filter(col("Prefijo_Plan") != col("IdProveedor_T").cast("string")).count()
print(f"Filas en F3 donde los primeros 5 dígitos de IdPlan_T NO coinciden con IdProveedor_T: {incons_prov}")

=== 5. INTEGRIDAD REFERENCIAL (CRUCE DE F3 CON F1, F2 Y F4) ===
F3 -> F1 (Áreas): 4979 filas en F3 (1097 IDs de área distintos) NO existen en F1.
F3 -> F2 (Beneficios): 2023 filas en F3 (107 IDs de beneficio distintos) NO existen en F2.

Conteo de IDs únicos en F4 tras homologar 'Copagado'->'Copago' y 'Coseguridad'->'Coseguro':
+---------------+----------+
|Tipo_Homologado|IDs_Unicos|
+---------------+----------+
|            NaN|         1|
|  SinTipoCopago|         1|
|         Copago|        15|
|       Coseguro|         6|
+---------------+----------+

F3 -> F4: Filas en F3 con IdCondicionDePagoCopago_T inexistente en F4: 0
F3 -> F4: Filas en F3 con IdCondicionDePagoCoseguro_T inexistente en F4: 0
Filas en F3 donde los primeros 5 dígitos de IdPlan_T NO coinciden con IdProveedor_T: 0


> **Análisis y conclusiones de Unicidad:**
> 1. **Colisión de llave primaria en `F4 (Condiciones de Pago)`:** Tras eliminar las 7 filas exactamente duplicadas (quedando 24 registros distintos para 23 IDs únicos), persiste una **violación de unicidad de llave primaria** en el `IdCondicionesDePago_T = 204`, el cual aparece dos veces con descripciones distintas: *"Copay per Stay after deductible"* y *"Copay per Day"*.
> 2. **Multiplicidad e inconsistencia severa de entidad en `F2 (Tipos de Beneficio)`:** Al descartar los 271 duplicados exactos quedan 578 filas distintas, pero solo existen **218 combinaciones únicas de `(IdTipoBeneficio_T, Fecha)`**. Esto significa que hay **360 filas en colisión de llave**: un mismo beneficio en un mismo año aparece registrado múltiples veces con atributos contradictorios en `EsEHB`, `EstaCubiertaPorSeguro`, `TieneLimiteCuantitativo` y exclusiones de desembolso (por ejemplo, el beneficio `ID = 5` en 2017 tiene 4 registros distintos que contradicen si es EHB o si está cubierto por seguro, y el `ID = 10` presenta valores contradictorios en su límite cuantitativo y unidad). Esto se debe a que en la fuente se desagregaron condiciones particulares de planes dentro del catálogo maestro de beneficios sin incluir un identificador de plan o variación.
> 3. **Unicidad de llave compuesta en `F1 (Áreas de Servicio)` y `F3 (Planes de Beneficio)`:** Una vez removidos los duplicados exactos de fila completa (43,573 en `F1` y 8,493 en `F3`) y filtrados los identificadores nulos/vacíos, **no existen colisiones de llave** (`0` colisiones en las 135,560 llaves compuestas válidas de `F1` `(IdAreaDeServicio_T, IdGeografia_T, Fecha)` y `0` colisiones en las 22,968 llaves compuestas válidas de `F3`). Es decir, el problema de unicidad en `F1` y `F3` corresponde estrictamente a duplicación exacta de registros en la extracción y a nulos en llaves.

> **Análisis y conclusiones de Consistencia (Intra-tabla, Reglas de Negocio e Integridad Referencial):**
> 1. **Inconsistencias intra-tabla en `F1 (Áreas de Servicio)`:**
>    * Se identificaron **6,177 registros con área geográfica negativa o cero** (llegando a `-24,707`), lo cual es físicamente imposible.
>    * Existen **6,142 registros con poblaciones absurdas superiores a 20 millones de habitantes por condado** (alcanzando $4.72 \times 10^{10}$), lo que altera la media poblacional y produce **6,182 registros donde la `Densidad` reportada no guarda relación matemática con `PoblacionAct / Area`**.
>    * Los **6,282 registros con `Fecha = 1800`** son exactamente los mismos cuyos últimos 4 dígitos de `IdAreaDeServicio_T` (que codifican el año real `2017` o `2018`) no coinciden con la columna `Fecha`. Esto demuestra que el año real puede reconstruirse extrayendo los últimos 4 dígitos de `IdAreaDeServicio_T`.
> 2. **Inconsistencias intra-tabla y de reglas de negocio en `F2` y `F3` (Límites cuantitativos y topes 2018):**
>    * En `F2`, aún homologando las etiquetas en distintos idiomas (`"Yes"`/`"Si"` y `"No"`/`"Nein"`), existen **33 registros con contradicción interna** entre tener límite cuantitativo y la presencia/ausencia de `UnidadDelLimite`.
>    * **Incumplimiento de la Regla de Negocio #2:** Al cruzar `F2` y `F3`, de las 19,322 asociaciones donde el beneficio sí exige límite cuantitativo, **14,471 registros (74.9%) incumplen la regla** al tener `cantidadLimite` nulo o $\le 0$. A la inversa, **4,774 registros** reportan una `cantidadLimite > 0` para beneficios que en `F2` figuran sin límite cuantitativo (producto de la colisión de llaves en `F2`, donde un mismo `IdTipoBeneficio_T` tiene filas con `"Yes"` y filas con `"No"`).
>    * **Incumplimiento parcial de la Regla de Negocio #5 y fechas anómalas en `F3`:** El coseguro máximo en 2018 cumple el tope del **100%** (`0` excesos), pero el copago máximo en 2018 alcanza **$3,500**, superando el límite de **$3,300** declarado por RaSA en **8 registros**. Además, se detectaron **2,052 registros con año `1920` (`1920-12-31`)** correspondientes a 212 planes únicos.
> 3. **Integridad referencial entre fuentes (`F3` hacia `F1`, `F2` y `F4`):**
>    * **Huérfanos hacia `F1`:** Existen **4,979 filas en `F3` (que involucran 1,097 `IdAreaDeServicio_T` distintos)** cuyo código de área de servicio **no existe en `F1`**. Esto explica por qué `F3` tiene 6,496 áreas distintas mientras que `F1` solo tiene 5,409 ($6,496 - 1,097 = 5,399$ áreas que sí cruzan).
>    * **Huérfanos hacia `F2`:** Existen **2,023 filas en `F3` (que involucran 107 `IdTipoBeneficio_T` distintos)** cuyo beneficio **no existe en el catálogo `F2`** ($285 - 107 = 178$ beneficios que sí cruzan).
>    * **Integridad hacia `F4` y homologación de tipos:** Todas las llaves foráneas de condiciones de pago en `F3` existen en `F4` (`0` huérfanos en copago y `0` en coseguro). Además, al homologar el error tipográfico `"Copagado" -> "Copago"`, **se alcanzan exactamente las 15 condiciones únicas de Copago** indicadas por RaSA en el enunciado (y 6 de Coseguro al homologar `"Coseguridad"`, más 1 `"SinTipoCopago"` y 1 `"NaN"`).
>    * **Consistencia `IdPlan_T` vs `IdProveedor_T`:** El 100% de los registros con `IdPlan_T` no vacío cumple que sus primeros 5 caracteres coinciden exactamente con `IdProveedor_T` (`0` inconsistencias), lo que confirma una regla sólida para auditar proveedores.

# cerrar sesion de pyspark

In [41]:
spark.stop()